<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **Launch Sites Locations Analysis with Folium**


Estimated time needed: **40** minutes


The launch success rate may depend on many factors such as payload mass, orbit type, and so on. It may also depend on the location and proximities of a launch site, i.e., the initial position of rocket trajectories. Finding an optimal location for building a launch site certainly involves many factors and hopefully we could discover some of the factors by analyzing the existing launch site locations.


In the previous exploratory data analysis labs, you have visualized the SpaceX launch dataset using `matplotlib` and `seaborn` and discovered some preliminary correlations between the launch site and success rates. In this lab, you will be performing more interactive visual analytics using `Folium`.


## Objectives


This lab contains the following tasks:
- **TASK 1:** Mark all launch sites on a map
- **TASK 2:** Mark the success/failed launches for each site on the map
- **TASK 3:** Calculate the distances between a launch site to its proximities

After completed the above tasks, you should be able to find some geographical patterns about launch sites.


Let's first import required Python packages for this lab:


In [362]:
#!pip  for mac
!pip  install folium
!pip  install wget
!pip  install pandas
!pip  install osmnx geopandas shapely

In [363]:
import folium
import wget
import pandas as pd
import osmnx as ox
import geopandas as gpd
from shapely.geometry import Point, LineString

In [364]:
# Import folium MarkerCluster plugin
from folium.plugins import MarkerCluster
# Import folium MousePosition plugin
from folium.plugins import MousePosition
# Import folium DivIcon plugin
from folium.features import DivIcon

If you need to refresh your memory about folium, you may download and refer to this previous folium lab:


[Generating Maps with Python](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/DV0101EN-3-5-1-Generating-Maps-in-Python-py-v2.0.ipynb)


## Task 1: Mark all launch sites on a map


First, let's try to add each site's location on a map using site's latitude and longitude coordinates


The following dataset with the name `spacex_launch_geo.csv` is an augmented dataset with latitude and longitude added for each site. 


In [365]:
# Download and read the `spacex_launch_geo.csv`
spacex_csv_file = wget.download('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv')
spacex_df=pd.read_csv(spacex_csv_file)

Now, you can take a look at what are the coordinates for each site.


In [366]:
# Select relevant sub-columns: `Launch Site`, `Lat(Latitude)`, `Long(Longitude)`, `class`
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df

,Launch Site,Lat,Long,class
0,CCAFS LC-40,28.562302,-80.577356,0
1,CCAFS SLC-40,28.563197,-80.576820,1
2,KSC LC-39A,28.573255,-80.646895,1
3,VAFB SLC-4E,34.632834,-120.610745,0


Above coordinates are just plain numbers that can not give you any intuitive insights about where are those launch sites. If you are very good at geography, you can interpret those numbers directly in your mind. If not, that's fine too. Let's visualize those locations by pinning them on a map.


We first need to create a folium `Map` object, with an initial center location to be NASA Johnson Space Center at Houston, Texas.


In [367]:
# Start location is NASA Johnson Space Center
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=10)

We could use `folium.Circle` to add a highlighted circle area with a text label on a specific coordinate. For example, 


In [368]:
# Create a blue circle at NASA Johnson Space Center's coordinate with a popup label showing its name
circle = folium.Circle(nasa_coordinate, radius=1000, color='#d35400', fill=True).add_child(folium.Popup('NASA Johnson Space Center'))
# Create a blue circle at NASA Johnson Space Center's coordinate with a icon showing its name
marker = folium.map.Marker(
    nasa_coordinate,
    # Create an icon as a text label
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'NASA JSC',
        )
    )
site_map.add_child(circle)
site_map.add_child(marker)

and you should find a small yellow circle near the city of Houston and you can zoom-in to see a larger circle. 


Now, let's add a circle for each launch site in data frame `launch_sites`


_TODO:_  Create and add `folium.Circle` and `folium.Marker` for each launch site on the site map


An example of folium.Circle:


`folium.Circle(coordinate, radius=1000, color='#000000', fill=True).add_child(folium.Popup(...))`


An example of folium.Marker:


`folium.map.Marker(coordinate, icon=DivIcon(icon_size=(20,20),icon_anchor=(0,0), html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'label', ))`


In [369]:

# For each launch site, add a Circle object based on its coordinate (Lat, Long) values. In addition, add Launch site name as a popup label
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)

for i in range(len(launch_sites_df)):
    Sitescoordinates = [launch_sites_df.iloc[i]['Lat'], launch_sites_df.iloc[i]['Long']]
    launch_site_name = launch_sites_df.iloc[i]['Launch Site']
    # Create a blue circle at NASA Johnson Space Center's coordinate with a popup label showing its name
    circle = folium.Circle(Sitescoordinates, radius=1000, color='#d35400', fill=True).add_child(folium.Popup(launch_site_name))
    # Create a blue circle at NASA Johnson Space Center's coordinate with a icon showing its name
    marker = folium.map.Marker(
        Sitescoordinates,
        # Create an icon as a text label
        icon=DivIcon(
            icon_size=(20,20),
            icon_anchor=(0,0),
            html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % launch_site_name,
        )
    )
    site_map.add_child(circle)
    site_map.add_child(marker)
site_map 

The generated map with marked launch sites should look similar to the following:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_markers.png">
</center>


Now, you can explore the map by zoom-in/out the marked areas
, and try to answer the following questions:
- Are all launch sites in proximity to the Equator line?
- Are all launch sites in very close proximity to the coast?

Also please try to explain your findings.


All of the launch location are within 28 to 35 Latitued from the Equator which is relatively close, it seems that all of these were diliberately placed at the Southern border of the US. From the map it can be inferred that all of those locations were also placed as far east or west or possible, which is by the coast.

# Task 2: Mark the success/failed launches for each site on the map


Next, let's try to enhance the map by adding the launch outcomes for each site, and see which sites have high success rates.
Recall that data frame spacex_df has detailed launch records, and the `class` column indicates if this launch was successful or not


In [370]:
spacex_df.tail(10)

,Launch Site,Lat,Long,class
46,KSC LC-39A,28.573255,-80.646895,1
47,KSC LC-39A,28.573255,-80.646895,1
48,KSC LC-39A,28.573255,-80.646895,1
49,CCAFS SLC-40,28.563197,-80.576820,1
50,CCAFS SLC-40,28.563197,-80.576820,1
51,CCAFS SLC-40,28.563197,-80.576820,0
52,CCAFS SLC-40,28.563197,-80.576820,0
53,CCAFS SLC-40,28.563197,-80.576820,0
54,CCAFS SLC-40,28.563197,-80.576820,1
55,CCAFS SLC-40,28.563197,-80.576820,0


In [371]:
spacex_df['Launch Site'].unique()

array(['CCAFS LC-40', 'VAFB SLC-4E', 'KSC LC-39A', 'CCAFS SLC-40'],
      dtype=object)

Next, let's create markers for all launch records. 
If a launch was successful `(class=1)`, then we use a green marker and if a launch was failed, we use a red marker `(class=0)`


Note that a launch only happens in one of the four launch sites, which means many launch records will have the exact same coordinate. Marker clusters can be a good way to simplify a map containing many markers having the same coordinate.


Let's first create a `MarkerCluster` object


In [372]:
marker_cluster = MarkerCluster()


_TODO:_ Create a new column in `launch_sites` dataframe called `marker_color` to store the marker colors based on the `class` value


In [373]:

# Apply a function to check the value of `class` column
# If class=1, marker_color value will be green
# If class=0, marker_color value will be red

# Function to assign color based on class value
def assign_marker_color(class_value):
    return 'green' if class_value == 1 else 'red'

# Create a new column in launch_sites_df called 'marker_color'
launch_sites_df['marker_color'] = launch_sites_df['class'].apply(assign_marker_color)

In [374]:
# Function to assign color to launch outcome
def assign_marker_color(launch_outcome):
    if launch_outcome == 1:
        return 'green'
    else:
        return 'red'
    
spacex_df['marker_color'] = spacex_df['class'].apply(assign_marker_color)
spacex_df.tail(10)

,Launch Site,Lat,Long,class,marker_color
46,KSC LC-39A,28.573255,-80.646895,1,green
47,KSC LC-39A,28.573255,-80.646895,1,green
48,KSC LC-39A,28.573255,-80.646895,1,green
49,CCAFS SLC-40,28.563197,-80.576820,1,green
50,CCAFS SLC-40,28.563197,-80.576820,1,green
51,CCAFS SLC-40,28.563197,-80.576820,0,red
52,CCAFS SLC-40,28.563197,-80.576820,0,red
53,CCAFS SLC-40,28.563197,-80.576820,0,red
54,CCAFS SLC-40,28.563197,-80.576820,1,green
55,CCAFS SLC-40,28.563197,-80.576820,0,red


_TODO:_ For each launch result in `spacex_df` data frame, add a `folium.Marker` to `marker_cluster`


In [375]:
import folium
from folium import Marker
from folium.plugins import MarkerCluster
from folium.features import DivIcon

# Create a base map centered around a starting point (e.g., the first launch site from launch_sites_df)
site_map = folium.Map(location=[launch_sites_df.iloc[0]['Lat'], launch_sites_df.iloc[0]['Long']], zoom_start=5)

# Create a MarkerCluster object
marker_cluster = MarkerCluster().add_to(site_map)

# Add circles and markers for each launch site
for i in range(len(launch_sites_df)):
    # Get the coordinates and launch site name
    site_coordinates = [launch_sites_df.iloc[i]['Lat'], launch_sites_df.iloc[i]['Long']]
    launch_site_name = launch_sites_df.iloc[i]['Launch Site']
    
    # Create a circle at the launch site coordinates with a popup label
    circle = folium.Circle(
        site_coordinates, 
        radius=1000, 
        color='#d35400', 
        fill=True, 
        fill_opacity=0.6
    ).add_child(folium.Popup(launch_site_name))
    
    # Create a marker displaying the launch site name
    marker = folium.map.Marker(
        site_coordinates,
        icon=DivIcon(
            icon_size=(20, 20),
            icon_anchor=(0, 0),
            html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % launch_site_name,
        )
    )
    
    # Add the circle and marker to the map
    site_map.add_child(circle)
    site_map.add_child(marker)
    
# Adding markers for launches (using your previous code snippet)
for index, record in spacex_df.iterrows():
    lat = record['Lat']  # Ensure using the correct column name
    lon = record['Long']
    launch_color = record['marker_color']  # Use the precomputed marker color
    
    marker = folium.Marker(
        location=[lat, lon],
        icon=folium.Icon(color='white', icon_color=launch_color)
    )
    marker_cluster.add_child(marker)
    
# Display the map
site_map

In [376]:
# Add marker_cluster to current site_map
"""site_map.add_child(marker_cluster)

# for each row in spacex_df data frame
# create a Marker object with its coordinate
# and customize the Marker's icon property to indicate if this launch was successed or failed, 
# e.g., icon=folium.Icon(color='white', icon_color=row['marker_color']
for index, record in spacex_df.iterrows():
    # TODO: Create and add a Marker cluster to the site map
    # marker = folium.Marker(...)
    marker_cluster.add_child(marker)

site_map """

"site_map.add_child(marker_cluster)\n\n# for each row in spacex_df data frame\n# create a Marker object with its coordinate\n# and customize the Marker's icon property to indicate if this launch was successed or failed, \n# e.g., icon=folium.Icon(color='white', icon_color=row['marker_color']\nfor index, record in spacex_df.iterrows():\n    # TODO: Create and add a Marker cluster to the site map\n    # marker = folium.Marker(...)\n    marker_cluster.add_child(marker)\n\nsite_map "

Your updated map may look like the following screenshots:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster.png">
</center>


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster_zoomed.png">
</center>


From the color-labeled markers in marker clusters, you should be able to easily identify which launch sites have relatively high success rates.


# TASK 3: Calculate the distances between a launch site to its proximities


Next, we need to explore and analyze the proximities of launch sites.


Let's first add a `MousePosition` on the map to get coordinate for a mouse over a point on the map. As such, while you are exploring the map, you can easily find the coordinates of any points of interests (such as railway)


In [377]:
# Add Mouse Position to get the coordinate (Lat, Long) for a mouse over on the map
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(
    position='topright',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=20,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter,
)

site_map.add_child(mouse_position)
site_map

Now zoom in to a launch site and explore its proximity to see if you can easily find any railway, highway, coastline, etc. Move your mouse to these points and mark down their coordinates (shown on the top-left) in order to the distance to the launch site.


You can calculate the distance between two points on the map based on their `Lat` and `Long` values using the following method:


In [378]:
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    # approximate radius of earth in km
    R = 6373.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    return distance

_TODO:_ Mark down a point on the closest coastline using MousePosition and calculate the distance between the coastline point and the launch site.


In [379]:
# Create a base map centered around the first launch site
site_map = folium.Map(location=[launch_sites_df.iloc[0]['Lat'], launch_sites_df.iloc[0]['Long']], zoom_start=5)

# Create a MarkerCluster object for launch sites
marker_cluster = MarkerCluster().add_to(site_map)

# Sample coastline coordinates (latitude, longitude) for demonstration
coastline_coordinates = [
    (28.5, -80.5),
    (28.6, -80.6),
    (28.4, -80.7),
    (28.55, -80.55),
    (34.4, -120.6)
]

# Add coastline points to the map
for coast_lat, coast_lon in coastline_coordinates:
    folium.Circle(
        location=[coast_lat, coast_lon],
        radius=100,
        color='blue',
        fill=True,
        fill_opacity=0.6,
        popup='Coastline Point'
    ).add_to(site_map)

# Add MousePosition to show coordinates
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(
    position='topright',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=5,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter,
)
site_map.add_child(mouse_position)

# Add circles and markers for each launch site
for i in range(len(launch_sites_df)):
    # Get the coordinates and launch site name
    site_coordinates = [launch_sites_df.iloc[i]['Lat'], launch_sites_df.iloc[i]['Long']]
    launch_site_name = launch_sites_df.iloc[i]['Launch Site']
    
    # Create a circle at the launch site coordinates with a popup label
    circle = folium.Circle(
        site_coordinates, 
        radius=1000, 
        color='#d35400', 
        fill=True, 
        fill_opacity=0.6
    ).add_child(folium.Popup(launch_site_name))
    
    # Create a marker displaying the launch site name
    marker = folium.map.Marker(
        site_coordinates,
        icon=DivIcon(
            icon_size=(20, 20),
            icon_anchor=(0, 0),
            html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % launch_site_name,
        )
    )

    # Add the circle and marker to the marker cluster
    marker_cluster.add_child(circle)
    marker_cluster.add_child(marker)

# Add an initial marker for the first launch site
folium.Marker(
    [launch_sites_df.iloc[0]['Lat'], launch_sites_df.iloc[0]['Long']],
    popup='Launch Site',
    icon=folium.Icon(color='green')
).add_to(site_map)

# Display the resulting map
site_map

In [380]:
# find coordinate of the closet coastline
# e.g.,: Lat: 28.56367  Lon: -80.57163
# distance_coastline = calculate_distance(launch_site_lat, launch_site_lon, coastline_lat, coastline_lon)


_TODO:_ After obtained its coordinate, create a `folium.Marker` to show the distance


In [381]:
# Fixed coordinates
launch_site_coord = (28.573255, -80.646895)
coastline_coord = (28.56367, -80.57163)

# Calculate distance using your predefined function
distance = calculate_distance(
    launch_site_coord[0], launch_site_coord[1],
    coastline_coord[0], coastline_coord[1]
)

# Initialize folium map centered between the two points
site_map = folium.Map(location=[28.5685, -80.609], zoom_start=12)

# Marker for the launch site
folium.Marker(
    location=launch_site_coord,
    popup="Launch Site",
    icon=folium.Icon(color='green')
).add_to(site_map)

# Marker for the coastline with distance label
folium.Marker(
    location=coastline_coord,
    icon=DivIcon(
        icon_size=(20, 20),
        icon_anchor=(0, 0),
        html='<div style="font-size: 12px; color:#d35400;"><b>{:.2f} KM</b></div>'.format(distance)
    )
).add_to(site_map)

# Draw line between launch site and coastline
folium.PolyLine(
    locations=[launch_site_coord, coastline_coord],
    color='blue',
    weight=2
).add_to(site_map)

# Show the map
site_map



In [382]:
# Create and add a folium.Marker on your selected closest coastline point on the map
# Display the distance between coastline point and launch site using the icon property 
# for example
# distance_marker = folium.Marker(
#    coordinate,
#    icon=DivIcon(
#        icon_size=(20,20),
#        icon_anchor=(0,0),
#        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(distance),
#        )
#    )
# Initialize minimum distance and closest coastline coordinate
#Fixed coordinates
launch_site_lat, launch_site_lon = 28.573255, -80.646895
coastline_lat, coastline_lon = 28.56367, -80.57163

# Calculate distance
distance = calculate_distance(launch_site_lat, launch_site_lon, coastline_lat, coastline_lon)

# Initialize folium map
site_map = folium.Map(location=[28.5685, -80.609], zoom_start=12)

# Add launch site marker
folium.Marker(
    location=[launch_site_lat, launch_site_lon],
    popup='Launch Site',
    icon=folium.Icon(color='green')
).add_to(site_map)

# Add coastline marker showing distance using DivIcon
distance_marker = folium.Marker(
    location=[coastline_lat, coastline_lon],
    icon=DivIcon(
        icon_size=(20, 20),
        icon_anchor=(0, 0),
        html='<div style="font-size: 12px; color:#d35400;"><b>{:10.2f} KM</b></div>'.format(distance),
    )
)
distance_marker.add_to(site_map)

# Draw line between launch site and coastline
folium.PolyLine(
    locations=[[launch_site_lat, launch_site_lon], [coastline_lat, coastline_lon]],
    color='blue',
    weight=2
).add_to(site_map)

# Display map
site_map


_TODO:_ Draw a `PolyLine` between a launch site to the selected coastline point


In [383]:
# Create a `folium.PolyLine` object using the coastline coordinates and launch site coordinate
# lines=folium.PolyLine(locations=coordinates, weight=1)
#site_map.add_child(lines)

# Prepare the coordinates for the PolyLine
"""coordinates = [
    (launch_site_lat, launch_site_lon),  # Launch site coordinates
    closest_coastline                     # Closest coastline coordinates
]

# Create a PolyLine object using the coordinates
lines = folium.PolyLine(locations=coordinates, weight=1, color='blue')

# Add the PolyLine to the map
site_map.add_child(lines) """

# Assuming launch_site_lat and launch_site_lon are the coordinates of your launch site
launch_site_lat = 34.7  # Example: For the purpose of illustration
launch_site_lon = -120.5 

# Example closest coastline coordinates already established from previous discussion
closest_coastline = (34.1, -120.0)  # Example coastline point

coordinates = [
    (launch_site_lat, launch_site_lon),  # Launch site coordinates
    closest_coastline                     # Closest coastline coordinates
]

# Create a PolyLine object using the coordinates
lines = folium.PolyLine(locations=coordinates, weight=2, color='blue')

# Add the PolyLine to the map
site_map.add_child(lines)

Your updated map with distance line should look like the following screenshot:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_distance.png">
</center>


_TODO:_ Similarly, you can draw a line betwee a launch site to its closest city, railway, highway, etc. You need to use `MousePosition` to find the their coordinates on the map first


A railway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/railway.png">
</center>


A highway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/highway.png">
</center>


A city map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/city.png">
</center>


In [384]:
# Create a marker with distance to a closest city, railway, highway, etc.
# Draw a line between the marker to the launch site
closest_highway = 28.56335, -80.57085
closest_railroad = 28.57206, -80.58525
closest_city = 28.10473, -80.64531

distance_highway = calculate_distance(launch_site_lat, launch_site_lon, closest_highway[0], closest_highway[1])
print('distance_highway =',distance_highway, ' km')
distance_railroad = calculate_distance(launch_site_lat, launch_site_lon, closest_railroad[0], closest_railroad[1])
print('distance_railroad =',distance_railroad, ' km')
distance_city = calculate_distance(launch_site_lat, launch_site_lon, closest_city[0], closest_city[1])
print('distance_city =',distance_city, ' km')


distance_highway = 3817.327521432381  km
distance_railroad = 3815.6697933094615  km
distance_city = 3828.7440444456597  km


In [ ]:
# closest highway marker
distance_marker = folium.Marker(
   closest_highway,
   icon=DivIcon(
       icon_size=(20,20),
       icon_anchor=(0,0),
       html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(distance_highway),
       )
   )
site_map.add_child(distance_marker)
# closest highway line
coordinates = [[launch_site_lat,launch_site_lon],closest_highway]
lines=folium.PolyLine(locations=coordinates, weight=1)
site_map.add_child(lines)

# closest railroad marker
distance_marker = folium.Marker(
   closest_railroad,
   icon=DivIcon(
       icon_size=(20,20),
       icon_anchor=(0,0),
       html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(distance_railroad),
       )
   )
site_map.add_child(distance_marker)
# closest rail line
coordinates = [[launch_site_lat,launch_site_lon],closest_railroad]
lines=folium.PolyLine(locations=coordinates, weight=1)
site_map.add_child(lines)

# closest city marker
distance_marker = folium.Marker(
   closest_city,
   icon=DivIcon(
       icon_size=(20,20),
       icon_anchor=(0,0),
       html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(distance_city),
       )
   )
site_map.add_child(distance_marker)
# closest city line
coordinates = [[launch_site_lat,launch_site_lon],closest_city]
lines=folium.PolyLine(locations=coordinates, weight=1)
site_map.add_child(lines)

After you plot distance lines to the proximities, you can answer the following questions easily:
- Are launch sites in close proximity to railways?
- Are launch sites in close proximity to highways?
- Are launch sites in close proximity to coastline?
- Do launch sites keep certain distance away from cities?

Also please try to explain your findings.


- Launch sites are typically located near the equator to take advantage of Earth’s rotational speed—approximately 30 km/s eastward—which reduces the fuel needed to reach orbit.
- They are positioned close to coastlines so launches can take place over the ocean, enhancing safety in two key ways: (1) crews can abort and attempt a water landing if needed, and (2) it reduces the risk of debris falling on populated or developed areas.
- Proximity to highways supports the efficient transportation of personnel and essential equipment to and from the site.
- Nearby railways are crucial for moving heavy cargo and materials required for space missions.
- Launch sites are purposefully situated away from densely populated cities to minimize potential hazards to large populations in the event of an accident.

# Next Steps:

Now you have discovered many interesting insights related to the launch sites' location using folium, in a very interactive way. Next, you will need to build a dashboard using Ploty Dash on detailed launch records.


## Authors


[Yan Luo](https://www.linkedin.com/in/yan-luo-96288783/)


### Other Contributors


Joseph Santarcangelo


## Change Log


|Date (YYYY-MM-DD)|Version|Changed By|Change Description|
|-|-|-|-|
|2021-05-26|1.0|Yan|Created the initial version|


Copyright © 2021 IBM Corporation. All rights reserved.
